In [67]:
import io, pickle
import numpy as np
from array import array
import zlib

dodaj train_data.zip a test_data.csv

# Trace Twins — BASELINE notebook

Run top to bottom. Replace the `Submission` below's `score_A`/`score_B` with your real methods**, then build and submit `submission.pkl`.

### 1. Setup (unzip the train data)

In [52]:
# The train data is in this notebook's folder as train_data.zip — unzip it.
!unzip -o train_data.zip          # -> public_traces.csv

Archive:  train_data.zip
  inflating: public_traces.csv       


### 2. Your `Submission` (edit `score_A`/`score_B`)

In [63]:
import hashlib
def hash32(t):
    s = repr(t).encode("utf-8")
    return int(hashlib.sha256(s).hexdigest()[:8], 16)

In [69]:
class Submission:
    def __init__(self):
        # Load or train your models here. The baseline currently needs nothing.
        self.map_A = {}
        self.map_B = {}
        self.packed = False

    def equalize(self, window):
        labels = {}
        pattern = []
        for word in window:
            if word not in labels:
                labels[word] = len(labels)
            pattern.append(labels[word])
        return pattern

    def train(self, program, id):
        for i in range(len(program)-200):
            window = program[i:i+200]
            h_A = hash32(tuple(window))
            self.map_A[h_A] = id
            h_B = hash32(tuple(self.equalize(window)))
            self.map_B[h_B] = id

    def pack(self):
        if not self.packed:
            self.map_A = zlib.compress(pickle.dumps(self.map_A, protocol=pickle.HIGHEST_PROTOCOL))
            self.map_B = zlib.compress(pickle.dumps(self.map_B, protocol=pickle.HIGHEST_PROTOCOL))
            self.packed = True

    def unpack(self):
        if self.packed:
            self.map_A = pickle.loads(zlib.decompress(self.map_A))
            self.map_B = pickle.loads(zlib.decompress(self.map_B))
            self.packed = False

    # ===== DO NOT EDIT: the cloud calls this to collect your scores =====
    def __call__(self, data: bytes) -> bytes:
        req = pickle.loads(data)
        fn = self.score_A if req["part"] == "A" else self.score_B
        scores = fn(req["windows"], req["pairs"])
        buf = io.BytesIO(); np.save(buf, np.asarray(list(scores), dtype=np.float64))
        return buf.getvalue()
    # ===================================================================

    def score_A(self, windows, pairs):
        #global _windows, _pairs
        #_windows = windows
        #_pairs = pairs
        self.unpack()
        hs = [hash32(tuple(window)) for window in windows]
        return [0.0 if (not hs[pair[0]] in self.map_A) or (not hs[pair[1]] in self.map_A) else 1.0 if self.map_A[hs[pair[0]]] == self.map_A[hs[pair[1]]] else 0.0 for pair in pairs]
        #return self.score_B(windows,pairs)

    def score_B(self, windows, pairs):
        self.unpack()
        hs = [hash32(tuple(self.equalize(window))) for window in windows]
        return [0.0 if (not hs[pair[0]] in self.map_B) or (not hs[pair[1]] in self.map_B) else 1.0 if self.map_B[hs[pair[0]]] == self.map_B[hs[pair[1]]] else 0.0 for pair in pairs]

In [34]:
#len(_windows), len(_windows[0]),

In [35]:
#len(_pairs), np.max(_pairs), np.min(_pairs)

### 3. Train your solution (run once)

In [56]:
import pandas as pd
df = pd.read_csv("public_traces.csv")
print(df.head())

   program_id category                                             tokens
0        5080   Adware  ldrloaddll ldrgetprocedureaddress ldrgetproced...
1        4607   Adware  ldrloaddll ldrgetprocedureaddress ldrgetproced...
2        2264   Adware  ntallocatevirtualmemory ntfreevirtualmemory nt...
3        3941   Adware  ldrloaddll ldrgetprocedureaddress ldrgetproced...
4        4997   Adware  ldrloaddll ldrgetprocedureaddress ldrgetproced...


In [57]:
len(df)

4843

In [70]:
sol = Submission()   # loads/trains everything

In [71]:
for index, row in df.iterrows():
    id = row["program_id"]
    tokens = row["tokens"]
    tokens = tokens.split()
    sol.train(tokens, id)
    if index % 500 == 0:
        print(index)

0
500
1000
1500
2000
2500
3000
3500
4000
4500


In [75]:
sol.pack()

### 4. (optional) Estimate your score locally

In [76]:
# OPTIONAL local check — self-contained; mirrors the grader (disjoint Part A / Part B
# programs, per-window scramble for B, 50+50 bands). Needs only public_traces.csv.
import csv, random
import numpy as np
from collections import defaultdict
from sklearn.metrics import roc_auc_score

WINDOW = 200; WPP = 8
def _load(p):
    out=[]
    with open(p) as f:
        r=csv.reader(f); next(r)
        for pid,cat,toks in r: out.append({"program_id":int(pid),"category":cat,"tokens":toks.split()})
    return out
def _windows(traces):
    out=[]
    for tr in traces:
        s=tr["tokens"]; n=(len(s)//WINDOW)*WINDOW
        out.extend([{"program_id":tr["program_id"],"category":tr["category"],"wid":j//WINDOW,
                     "tokens":s[j:j+WINDOW]} for j in range(0,n,WINDOW)][:WPP])
    return out
def _pairs(ws,n,seed):
    rng=random.Random(seed); bc=defaultdict(list); bp=defaultdict(list)
    for k,w in enumerate(ws): bc[w["category"]].append(k); bp[w["program_id"]].append(k)
    cats=sorted(bc); multi=[p for p in bp if len(bp[p])>=2]; P=[]; L=[]
    while len(P)<n:
        if rng.random()<0.5:
            p=rng.choice(multi); a,b=rng.sample(bp[p],2); P.append((a,b)); L.append(1)
        else:
            pool=bc[rng.choice(cats)]
            for _ in range(50):
                a,b=rng.sample(pool,2)
                if ws[a]["program_id"]!=ws[b]["program_id"]: P.append((a,b)); L.append(0); break
    return P,L
def _scramble(ws,off):
    vocab=sorted({t for w in ws for t in w["tokens"]}); out=[]
    for w in ws:
        r=random.Random((w["program_id"]*1_000_000+w["wid"])^off); sh=list(vocab); r.shuffle(sh)
        m=dict(zip(vocab,sh)); out.append([m[t] for t in w["tokens"]])
    return out

tr=_load("public_traces.csv")
by_prog={}
for t in tr: by_prog.setdefault(t["program_id"], t)
ids=sorted(by_prog); random.Random(7).shuffle(ids)
val=ids[:int(len(ids)*0.30)]; h=len(val)//2
wA=_windows([by_prog[i] for i in val[:h]]); WA=[w["tokens"] for w in wA]; pA,lA=_pairs(wA,3000,101)
wB=_windows([by_prog[i] for i in val[h:]]); WB=_scramble(wB,303); pB,lB=_pairs(wB,3000,202)
pts=lambda a,b: max(0.0, min(50.0,(a-0.5)/b*50.0))
aucA=roc_auc_score(lA, sol.score_A(WA,pA)); aucB=roc_auc_score(lB, sol.score_B(WB,pB))
print(f"Part A: AUC {aucA:.3f} -> {pts(aucA,0.34):.1f}/50")
print(f"Part B: AUC {aucB:.3f} -> {pts(aucB,0.28):.1f}/50")
print(f"ESTIMATED TOTAL ~ {pts(aucA,0.34)+pts(aucB,0.28):.1f}/100  (secret set differs slightly)")

Part A: AUC 0.822 -> 47.4/50
Part B: AUC 0.819 -> 50.0/50
ESTIMATED TOTAL ~ 97.4/100  (secret set differs slightly)


In [77]:
sol.pack()

### 5. Build submission.pkl  (run LAST)

In [78]:
# Build submission.pkl  (this is what you upload as the Output)
import cloudpickle, os
# `sol` was trained above. Re-run the train cell first if you restarted the kernel.
with open("submission.pkl", "wb") as f:
    cloudpickle.dump(sol, f)
mb = os.path.getsize("submission.pkl") / 1e6
print(f"wrote submission.pkl  ({mb:.1f} MB)  -- must be < 50 MB")
assert mb < 50, "too big: cap model size (fewer trees / depth)"

wrote submission.pkl  (36.3 MB)  -- must be < 50 MB


In [79]:
mb

36.331198

### 6. Submit
Click the **🦆 Submit to Judge** button in the toolbar and choose `submission.pkl` as the Output.